## A Critical Look at Cuckoo Search

This lab explores the common issue of seemingly new optimization algorithms that may not offer genuinely novel ideas. We'll first examine the original Cuckoo Search paper and then critically analyze it using the paper "An analysis of why cuckoo search does not bring any novel ideas to optimization." This exercise builds on our previous discussions of evolutionary algorithms like Differential Evolution and the pitfalls of poor research in fields like Neuroevolution, focusing here on the problem of redundant concepts in optimization.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from typing import Callable
from dataclasses import dataclass
from math import gamma

### Optimization Problems

This cell defines three common benchmark functions, Sphere, Rosenbrock, and Rastrigin, used to test optimization algorithms. We also used these functions earlier to evaluate Adam, Momentum, and CMA-ES.


In [2]:
def sphere(x: np.ndarray) -> float:
    return float(np.sum(x**2))


def rosenbrock(x: np.ndarray) -> float:
    return float(np.sum(100.0 * (x[1:] - x[:-1] ** 2.0) ** 2.0 + (1.0 - x[:-1]) ** 2.0))


def rastrigin(x: np.ndarray) -> float:
    A: float = 10.0
    return float(A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x)))


BOUNDS = [(-5, 5), (-5, 5)]

### Visualizing Search Dynamics


In [10]:
def animate_cs(
    func: Callable[[np.ndarray], float],
    history: list[np.ndarray],
    bounds: list[tuple[float, float]] = BOUNDS,
    frames: int | None = None,
    filename: str = "cs_animation.gif",
) -> None:
    """
    Creates and saves a GIF showing how the CS population moves over generations.
    """
    if frames is None:
        frames = len(history)

    assert len(bounds) == 2, (
        "This function only supports 2D visualization (expected 2 bounds)."
    )
    x_bounds = (bounds[0][0], bounds[0][1])
    y_bounds = (bounds[1][0], bounds[1][1])

    x = np.linspace(x_bounds[0], x_bounds[1], 200)
    y = np.linspace(y_bounds[0], y_bounds[1], 200)
    X, Y = np.meshgrid(x, y)
    coords = np.vstack([X.ravel(), Y.ravel()]).T
    Z = np.array([func(pt) for pt in coords]).reshape(X.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    contour = ax.contourf(X, Y, Z, levels=20, cmap="viridis")
    fig.colorbar(contour, ax=ax)

    # Create the scatter once and update it each frame
    scatter = ax.scatter([], [], s=20, color="red")

    def init():
        scatter.set_offsets(np.empty((0, 2)))  # 2D empty array
        return (scatter,)

    def update(i: int):
        ax.set_title(f"Generation {i}")
        pop = history[i]
        scatter.set_offsets(pop)
        return (scatter,)

    ax.set_xlim(x_bounds[0], x_bounds[1])
    ax.set_ylim(y_bounds[0], y_bounds[1])

    anim = animation.FuncAnimation(
        fig, update, init_func=init, frames=frames, interval=200, blit=True
    )

    writer = animation.PillowWriter(fps=5)
    anim.save(filename, writer=writer)
    plt.close(fig)
    print(f"Animation saved to {filename}")


### Exercise 1

Read [Cuckoo Search via Levy Flights](https://arxiv.org/pdf/1003.1594) with particular attention to Sections 2, 3, and 4. The primary focus for this exercise is Figure 1, which outlines the core pseudocode of the algorithm.

Your task is to implement the Cuckoo Search algorithm, using Figure 1 as your main reference. As the pseudocode is relatively high-level and lacks
implementation details, you are encouraged to adopt a straightforward approach in your implementation.

**Action Item:** Document any ambiguities or unclear aspects you encounter in the algorithm description or pseudocode.

> Note: If you find the implementation too challenging or feel stuck, you may proceed directly to Exercise 2.


In [4]:
@dataclass
class CSResult:
    best_vector: np.ndarray
    best_value: float
    history: list[np.ndarray]  # History of populations for animation


class LevyFlight:
    def __init__(self, beta: float = 1.5):
        self.beta = beta

    def __call__(self, size: int) -> np.ndarray:
        sigma_u = (
            gamma(1 + self.beta)
            * np.sin(np.pi * self.beta / 2)
            / (gamma((1 + self.beta) / 2) * self.beta * 2 ** ((self.beta - 1) / 2))
        ) ** (1 / self.beta)
        sigma_v = 1
        u = np.random.normal(0, sigma_u, size)
        v = np.random.normal(0, sigma_v, size)
        step = u / np.abs(v) ** (1 / self.beta)
        return step


def cuckoo_search(
    func: Callable[[np.ndarray], float],
    bounds: list[tuple[float, float]] = BOUNDS,
    pop_size: int = 50,
    alpha: float = 1.0,
    beta: float = 1.5,
    p: float = 0.25,
    max_gen: int = 100,
) -> CSResult:
    """
    Implements the Cuckoo Search algorithm for global optimization.

    Parameters:
        func: Objective function to minimize. Takes a numpy array and returns a float.
        bounds: list of (min, max) pairs for each dimension.
        pop_size: Number of individuals in the population.
        alpha: Step-size scaling factor controlling the overall scale of the Lévy flights.
        beta: Exponent parameter for the Lévy distribution (typically in (1, 3]) that
            influences the heavy-tailed step-length distribution.
        p: Probability (in [0, 1]) that a host bird discovers an alien egg and abandons
            the corresponding nest—i.e., fraction of worse nests to be replaced each generation.
        max_gen: Maximum number of generations to evolve.
    """
    dimensions = len(bounds)
    lower_bounds = np.array([b[0] for b in bounds])
    upper_bounds = np.array([b[1] for b in bounds])
    history = []

    # --- TODO: Exercise 1 - Initialization ---
    # Initialize the population: Create pop_size individuals.
    # Each individual is a vector of 'dimensions' length.
    # Use uniform distribution to sample the initial population.
    # --- End Exercise 1 ---

    population = np.random.uniform(lower_bounds, upper_bounds, (pop_size, dimensions))
    cuckoo = np.random.uniform(lower_bounds, upper_bounds, dimensions)

    for generation in range(max_gen):
        # Get a cuckoo from the Lévy flight distribution
        cuckoo = cuckoo + alpha * LevyFlight(beta)(dimensions)

        # Evaluate new cuckoo
        cuckoo_fitness = func(cuckoo)
        random_nest = np.random.randint(0, len(population))
        random_nest_fitness = func(population[random_nest])

        if cuckoo_fitness < random_nest_fitness:
            population[random_nest] = cuckoo

        # Remove fraction of worse nests
        fraction_to_remove = int(p * len(population))
        for i in range(1, fraction_to_remove + 1):
            population[-i] = np.random.uniform(lower_bounds, upper_bounds, dimensions)

        # Sort the population based on fitness
        fitness = np.array([func(ind) for ind in population])
        sorted_indices = np.argsort(fitness)
        population = population[sorted_indices]
        best_vector = population[0]

        # Store the best value found so far
        best_value = func(best_vector)

        # Store the history of populations for animation
        if generation == 0:
            history = [population.copy()]
        else:
            history.append(population.copy())

    return CSResult(best_vector=best_vector, best_value=best_value, history=history)

### Test implemented CS


In [5]:
result = cuckoo_search(sphere, bounds=BOUNDS, pop_size=50)
result.best_vector, result.best_value

(array([-0.12678692,  0.03733262]), 0.017468646708277663)

### Experiments

Run CS on all three problems: Sphere, Rosenbrock and Rastrigin. For each problem:

- Visualize the population dynamics over time to illustrate how the search space is explored and exploited.


In [11]:
for problem in [sphere, rosenbrock, rastrigin]:
    result = cuckoo_search(problem, bounds=BOUNDS, pop_size=50)
    animate_cs(problem, result.history, filename=f"{problem.__name__}_animation.gif")

Animation saved to sphere_animation.gif
Animation saved to rosenbrock_animation.gif
Animation saved to rastrigin_animation.gif


### Exercise 2

Read [An analysis of why cuckoo search does not bring any novel ideas to optimization](https://www.sciencedirect.com/science/article/pii/S0305054822000442). Focus particularly on Sections 2 and 3. Section 2.3 provides a detailed description of the implemented Cuckoo Search algorithm. Carefully analyze and compare it with your own, highlighting any differences in assumptions, parameter settings, or algorithmic structure.


In the original paper, there was no mention of a for loop to iterate over all solutions in the population in order to perform perturbations. There was no mention of comparing the solutions pair-wise to determine which one is better, rather just randomly choose one example and compair it to a new solution. Also, it wasn't said that there need to be 2 solution vectors, one with discarded solutions and second with the remaining that is then used to for recombination of the new solution vector.


### Exercise 3

Read the Introduction of “An analysis of why cuckoo search does not bring any novel ideas to optimization.” Identify and outline three criteria proposed by the authors for evaluating the underlying metaphor of the algorithm. Critically reflect on these criteria, do you find them appropriate and sufficient? Can you suggest any additional criteria or alternative perspectives that might enrich the evaluation?


The proposed criteria are:

- **Usefulness** Does the metaphor bring useful concepts to solve optimization problems?
- **Novelty** Were the concepts brought by the metaphor new in the field of stochastic optimization at the time when they were proposed?
- **Sound motivation** Is there a sound motivation to use the metaphor?

Well, to be honest, I must say I agree with the 2 first criteria. Don't know what the researchers mean by sound motivation, but I think that if something is both novel and useful, then it is sound enough to be explored. And I think that if we're talking about geeks' (passionate people) implementation of evolutionary algorithms, these 2 reasons are enough. If we're talking about scientific research combined with practical applications, then I think that another criterion should be:

- **Applicability** If the algorithm is not applicable to real-world problems, then it is not useful, no matter how novel and sound it is.

and last one:

- **profitability** If the algorithm is computationally profitable


### Exercise 4

Read Section 4 of “An analysis of why cuckoo search does not bring any novel ideas to optimization.” Explain the main criticisms they raise against the Cuckoo Search algorithm. What fundamental issues do they identify, and how do these undermine the algorithm's novelty?


The researches compare Cuckoo Search (CS) to Evolution Strategy (ES). They identify that:

- **parental selection** both algorithms use the same parental selection mechanism
- **survival selection** CS uses the same survival selection mechanism as ES
- **mutation** While cuckoo search uses the Lévy distribution, ES strategies have used a number of different types of distribution: the Gaussian distribution, that was used in the original algorithm; the Cauchy distribution, that was introducedin 1994; and the Lévy distribution introduced in 2002. Therefore, the mutation operator used by cuckoo search is exactly the same as in ES.
- **recombination** In cuckoo search, recombination differs in two aspects from the way this operator is traditionally implemented in ES. First, the specific recombination mechanism implemented in cuckoo search was not defined in the context of ES, but in the one of differential evolution (DE). Second, it is applied at the end of the algorithm and not at the beginning as it is normally done in ES. However, the two algorithms become equivalent starting from iteration 2. As cuckoo search does not follow the normal order in which the four main components of evolutionary algorithms are used, solutions have to be evaluated twice at each iteration of the while loop, which results in wasting computational time.

The issues not only undermine the algorithm's novelty, but also stress its lower performance.


### Exercise 5

Analyze Figures 5 and 8 from [Large-scale Benchmarking of Metaphor-based Optimization Heuristics](https://arxiv.org/pdf/2402.09800). In these figures, Cuckoo Search is denoted as CS. Evaluate its performance relative to the CMA-ES variant (bipop) and Differential Evolution (DE). How does CS compare to these well-established algorithms in terms of optimization performance? Furthermore, critically consider whether performance alone is a sufficient criterion for evaluating optimization algorithms. What other factors should be taken into account?


Surprisingly, CS performs better than most of the benchmarked algorithms. The figure 5 suggests that if CS is good at optimising a function, it will do well starting from the first iterations. We can say same about DE, but not about CMA-ES. CMA-ES seems to perform well but only if given a sufficient budget.
The figure 8 shows that on average, CS scores the best in the benchmarked problems. Other DE variants (DE, JADE, SHADE, LSHADE) perform a bit worse, but maintain higher distance from the baseline algorithms.

As the researchers said: "A complementary aspect of understanding an algorithm’s strengths is identifying whether it shows some performance characteristics which are not present in established algorithms. This might suggest that an algorithm contains useful new ideas or a way of combining existing ideas in a beneficial manner.(...) In many cases, algorithms can be widely used for years, to then be found to be equivalent to an existing algorithm with modified naming schemes. **An example of this is the Cuckoo Search algorithm (CS), which performed especially well in this paper. However, it has been shown that CS contains no novelty, and is simply a reformulation of an existing ES variant**."

What is still surprising to me is that in “An analysis of why cuckoo search does not bring any novel ideas to optimization.” it was stated that the CS algorithm might perform worse because of the double calculation of recombination phase, but in the figures it performs better than DE and CMA-ES.

As stated before, in my opinion, performance is enough criterion to evaluate an algorithm. Maybe we shouldn't state that we've come up with something new and a novel idea, rather just say that we have a new implementation of an existing algorithm that performs better than the previous ones. And the more complex and resource heavy the problem we want to optimise is, the more performance matters.
